In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora/adapter_model.safetensors
/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora/adapter_config.json
/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora/README.md
/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora/tokenizer.json
/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora/tokenizer_config.json
/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora/chat_template.jinja
/kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


# EDA

We load the train set to understand:
- sentiment labels
- distribution of samples across languages
- distribution of sentiments across languages
- Average sentence length per sample

In [3]:
df = pd.read_csv('/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID        900 non-null    int64 
 1   sentence  900 non-null    object
 2   label     900 non-null    object
 3   language  900 non-null    object
dtypes: int64(1), object(3)
memory usage: 28.3+ KB


The dataset has only non-null values, which means we don't need to drop any records from the dataset.

In [5]:
df['label'].unique()

array(['Negative', 'Positive'], dtype=object)

In [6]:
df['language'].unique()

array(['pa', 'ml', 'bn', 'gu', 'as', 'ur', 'mr', 'ta', 'hi', 'te', 'or',
       'kn', 'bd'], dtype=object)

Checking distribution of samples across labels and languages

In [7]:
df.groupby('label')['ID'].count()

label
Negative    444
Positive    456
Name: ID, dtype: int64

There is an approximately even split of positive and negative labels in the data with slightly more positive samples.

In [8]:
df.groupby('language')['ID'].count()

language
as    71
bd    71
bn    65
gu    69
hi    74
kn    66
ml    68
mr    67
or    72
pa    72
ta    76
te    63
ur    66
Name: ID, dtype: int64

In [9]:
print(df.groupby(['language', 'label'])['ID'].count())

language  label   
as        Negative    35
          Positive    36
bd        Negative    35
          Positive    36
bn        Negative    31
          Positive    34
gu        Negative    34
          Positive    35
hi        Negative    35
          Positive    39
kn        Negative    33
          Positive    33
ml        Negative    33
          Positive    35
mr        Negative    36
          Positive    31
or        Negative    35
          Positive    37
pa        Negative    38
          Positive    34
ta        Negative    37
          Positive    39
te        Negative    29
          Positive    34
ur        Negative    33
          Positive    33
Name: ID, dtype: int64


Checking average number of words per sample (using whitespace as a separator)

In [10]:
token_mean = df['sentence'].str.split().str.len().mean()

In [11]:
token_mean

np.float64(21.246666666666666)

In [12]:
df['word_count'] = df['sentence'].str.split().str.len()
df.groupby('language')['word_count'].agg('mean')

language
as    22.070423
bd    20.901408
bn    21.123077
gu    21.507246
hi    27.324324
kn    17.090909
ml    16.602941
mr    21.447761
or    20.125000
pa    24.319444
ta    16.078947
te    16.761905
ur    30.606061
Name: word_count, dtype: float64

Urdu and Hindi emerge as verbose languages, while Dravidian languages have fewer whitespace-separated words. 

Next, we will use the Gemma tokenizer and count token lengths. 

In [13]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(token=secret_value_0)

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

df["tokens"] = df["sentence"].apply(
    lambda x: tokenizer.encode(x)
)
df['token_lengths'] = df["tokens"].apply(
    lambda x: len(x)
)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [15]:
print(df['token_lengths'].mean())

52.44222222222222


In [16]:
print(df.groupby('language')['token_lengths'].agg(['mean', 'max', 'min']))

               mean  max  min
language                     
as        60.000000  167   10
bd        66.985915  149   11
bn        35.353846   79    5
gu        48.695652  120   12
hi        38.486486  152    9
kn        52.484848  135    9
ml        53.235294  184   13
mr        42.597015  121    7
or        85.236111  196   12
pa        67.416667  173   17
ta        36.881579  130    7
te        48.285714  111   10
ur        43.969697  151   12


## Conclusion
- Data is largely balanced across sentiments and language
- While some languages get split into more tokens, each sentence is short enough to fit into normal context lengths.
- All the languages are quite fertile, with more tokens than whitespace-separated words. Some languages such as Assamese, Bodo and the Dravidian languages are slightly more fertile than the others.  

In [17]:
def tokenize(example):
    example = tokenizer(example['sentence'],padding=False,truncation=True)
    return example

In [18]:
from datasets import Dataset
ds = Dataset.from_pandas(df)

In [19]:
ds

Dataset({
    features: ['ID', 'sentence', 'label', 'language', 'word_count', 'tokens', 'token_lengths'],
    num_rows: 900
})

In [20]:
labelmap = {'Positive': 1, 'Negative': 0}
ds = ds.map(lambda x: {"label": labelmap[x["label"]]})

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [21]:
ds_tokenized = ds.map(tokenize, batched = True, num_proc = 12, remove_columns = ['word_count', 'tokens', 'token_lengths', 'language', 'sentence'])

Map (num_proc=12):   0%|          | 0/900 [00:00<?, ? examples/s]

In [22]:
ds_split = ds_tokenized.train_test_split(test_size=0.1,seed=42)
print(ds_split)

DatasetDict({
    train: Dataset({
        features: ['ID', 'label', 'input_ids', 'attention_mask'],
        num_rows: 810
    })
    test: Dataset({
        features: ['ID', 'label', 'input_ids', 'attention_mask'],
        num_rows: 90
    })
})


In [23]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer, padding=True)

In [24]:
# from transformers import AutoModelForSequenceClassification
# import torch
# model = AutoModelForSequenceClassification.from_pretrained("google/gemma-3-1b-it",num_labels=2,
#                                                            pad_token_id=tokenizer.eos_token_id)
# from peft import LoraConfig, TaskType, LoraModel
# lora_config = LoraConfig(
#     r=16,
#     target_modules=["q_proj", "v_proj"],
#     task_type=TaskType.SEQ_CLS,
#     inference_mode=False,
#     lora_alpha=32,
#     lora_dropout=0.05
# )
# from peft import get_peft_model
# lora_model = get_peft_model(model, lora_config)
# lora_model.print_trainable_parameters()
# print(lora_model)

In [25]:
import evaluate

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [26]:
# from transformers import TrainingArguments, Trainer
# training_args = training_args = TrainingArguments( output_dir='gemma_nlp_ft_lora',
#                                   eval_strategy="steps",
#                                   eval_steps=200,
#                                   num_train_epochs=3,
#                                   per_device_train_batch_size=4,
#                                   per_device_eval_batch_size=4,
#                                   bf16=False,
#                                   fp16=False,
#                                   tf32=False,
#                                   gradient_accumulation_steps=1,
#                                   adam_beta1=0.9,
#                                   adam_beta2=0.999,
#                                   learning_rate=2e-4,
#                                   weight_decay=0.01,
#                                   logging_dir='logs',
#                                   logging_strategy="steps",
#                                   logging_steps = 100,
#                                   save_steps=600,
#                                   save_total_limit=10,
#                                   report_to='none',
#                                 )

In [27]:
# trainer = Trainer(model=lora_model,
#                   args = training_args,
#                  train_dataset=ds_split["train"],
#                  eval_dataset=ds_split["test"],
#                  compute_metrics=compute_metrics,
#                  data_collator = data_collator)

In [28]:
#results = trainer.train()

In [29]:
# results
# TrainOutput(global_step=306, training_loss=0.8402878349902583, metrics={'train_runtime': 332.013, 'train_samples_per_second': 7.319, 'train_steps_per_second': 0.922, 'total_flos': 1027635684354048.0, 'train_loss': 0.8402878349902583, 'epoch': 3.0})

In [30]:
# trainer.model.save_pretrained("gemma3_nlp_lora")
# tokenizer.save_pretrained("gemma3_nlp_lora")

In [31]:
# !zip -r gemma3_nlp_lora.zip gemma3_nlp_lora

# Predictions for Test Set

In [32]:
from transformers import AutoModelForSequenceClassification
from peft import PeftModel
model = AutoModelForSequenceClassification.from_pretrained("google/gemma-3-1b-it", num_labels=2,
                                                           pad_token_id = tokenizer.eos_token_id)

lora_model = PeftModel.from_pretrained(
    model,
    "/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora"
)
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/krithikakannan13/basic-peft/gemma3_nlp_lora")

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma3TextForSequenceClassification LOAD REPORT from: google/gemma-3-1b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
from transformers import Trainer
trainer = Trainer(model = lora_model, compute_metrics=compute_metrics,
                  data_collator = data_collator)

In [34]:
test_df = pd.read_csv('/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv')
test_df

,ID,sentence,language
0,550,നന്നായി പരിപാലിക്കപ്പെടുന്നതും വൃത്തിയുള്ളതുമാ...,ml
1,397,ക്യാമറ ലെൻസിൽ നന്നായി ചേരുന്നതിനാൽ പൊടിയിൽ നിന...,ml
2,757,ਕੂਲਰ ਵਧੀਆ ਕੈਸਟਰ ਦੇ ਪਹੀਏ ਨਾਲ ਫਿੱਟ ਹੁੰਦਾ ਹੈ। ਹਾਲ...,pa
3,407,"రాకెట్ తేలికగా ఉన్నప్పటికీ, యోనెక్స్ రాకెట్లతో...",te
4,294,गासै रोखोमनि दामनाय आदब स्मार्टफनफोरजोंबो गोरो...,bd
...,...,...,...
95,748,"छोटी, आरामदायी चांगले पॅडिंग असलेली आणि वॉटरप्...",mr
96,73,آئی کال (iKall) کے ٹاور اسپیکر وائی فائی کے ذر...,ur
97,942,মেশিন গরম হয়ে যায়।,bn
98,649,તે હોલ્ડિંગ છેડે સરસ ગાદી સાથેનો ખૂબ જ યોગ્ય પ...,gu


In [35]:
test_ds = Dataset.from_pandas(test_df)

In [36]:
test_tokenized = test_ds.map(tokenize, batched = True, num_proc = 12, remove_columns = ['sentence', 'language'])

Map (num_proc=12):   0%|          | 0/100 [00:00<?, ? examples/s]

In [37]:
test_tokenized

Dataset({
    features: ['ID', 'input_ids', 'attention_mask'],
    num_rows: 100
})

In [38]:
preds = trainer.predict(test_tokenized)

In [39]:
labels = preds.predictions.argmax(axis=-1)

In [40]:
submission = pd.DataFrame({
    "ID": test_df['ID'],
    "label": labels
})

submission.to_csv("submission.csv", index=False)